# Damoxingtest GPT-2 daily training

This private Kaggle notebook requests a GPU, verifies the actual device in the training script, fine-tunes a Chinese GPT-2 with LoRA, and writes metrics, logs, samples, and an adapter under /kaggle/working/training_output.

The project rules are: protect the user; maintain humanity. The run is an experiment, not a safety certification.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path

repo = Path('/kaggle/temp/damoxingtest')
output_dir = Path('/kaggle/working/training_output')
status_path = Path('/kaggle/working/training_status.json')
if output_dir.exists():
    shutil.rmtree(output_dir)
if status_path.exists():
    status_path.unlink()
output_dir.mkdir(parents=True, exist_ok=True)
print('Notebook start UTC:', datetime.now(timezone.utc).isoformat())

try:
    if repo.exists():
        shutil.rmtree(repo)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/mjw3177109/damoxingtest.git', str(repo)
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', str(repo / 'requirements-kaggle.txt')
    ], check=True)
    command = [
        sys.executable, str(repo / 'scripts' / 'train_gpt2_lora.py'),
        '--data', str(repo / 'data' / 'safety_humanity_seed.jsonl'),
        '--output-dir', str(output_dir),
        '--require-gpu', '--require-t4'
    ]
    print('Launching training command:', ' '.join(command))
    result = subprocess.run(command, cwd=str(repo), check=False)
    if result.returncode != 0:
        if not status_path.exists():
            status_path.write_text(json.dumps({
                'status': 'failed',
                'error': 'Training script exited with code ' + str(result.returncode)
            }, ensure_ascii=False, indent=2), encoding='utf-8')
        print('TRAINING_FAILED with exit code', result.returncode)
    else:
        print('Training script returned success.')
except Exception as exc:
    failure = {
        'status': 'failed',
        'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        'error_type': type(exc).__name__,
        'error': str(exc),
        'traceback': traceback.format_exc()
    }
    status_path.write_text(json.dumps(failure, ensure_ascii=False, indent=2), encoding='utf-8')
    print('NOTEBOOK_SETUP_FAILED:', type(exc).__name__, str(exc))
    traceback.print_exc()

if status_path.exists():
    print('Training status:', status_path.read_text(encoding='utf-8'))
print('Notebook end UTC:', datetime.now(timezone.utc).isoformat())